# 🎨 Effigy - Remote GPU Server on Google Colab

This notebook runs **Effigy** (the fully local 2D/3D AI generation suite) on a Google Colab GPU instance and exposes it via a secure **Cloudflare Tunnel** so you can access the full generation UI from any device.

### Instructions:
1. **Enable GPU**: Click **Runtime** → **Change runtime type** → select **L4 GPU** (or T4 GPU).
2. **Run All Cells**: Click **Runtime** → **Run all** (or Shift+Enter through each cell).
3. **Open Effigy**: Click the `*.trycloudflare.com` link printed at the bottom of Step 4.

In [ ]:
# Step 1: Verify NVIDIA GPU Allocation
!nvidia-smi

In [ ]:
# Step 2: Install uv, clone Effigy, install dependencies, and build UI
import os, sys

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# Clone repository if not already present
if not os.path.exists("/content/Effigy"):
    !git clone https://github.com/SREphoto/Effigy.git /content/Effigy

%cd /content/Effigy

# Sync environment
!uv sync --frozen
!UV_PROJECT_ENVIRONMENT=.venv-ml uv sync --frozen --group ml

# Install pinned headless Blender & gltfpack
!uv run python -m server.blender_install
!uv run python -m server.gltfpack_install

# Build the UI frontend so the server serves it
!cd /content/Effigy/ui && npm install && npm run build

In [ ]:
# Step 2.1: (Recommended) Connect Google Drive to persist all creations across Colab resets
from google.colab import drive
import os

drive.mount('/content/drive')

# Link Effigy library directly into Google Drive
drive_lib = "/content/drive/MyDrive/Effigy/library"
os.makedirs(drive_lib, exist_ok=True)

if os.path.isdir("/content/Effigy/library") and not os.path.islink("/content/Effigy/library"):
    !cp -ru /content/Effigy/library/* /content/drive/MyDrive/Effigy/library/ 2>/dev/null || true
    !rm -rf /content/Effigy/library

!ln -sfn /content/drive/MyDrive/Effigy/library /content/Effigy/library
print("\n✅ Google Drive linked! All 2D and 3D creations persist in MyDrive/Effigy/library")


In [ ]:
# Step 2.5: Compile & Install Linux CUDA Extensions (cumesh, flex_gemm, o_voxel)
# (Required for 3D generation, Trellis.2, Remeshing, UV Unwrapping, and Normal Baking on Linux)
import os, subprocess

os.environ["PATH"] = f"/content/Effigy/.venv-ml/bin:/root/.local/bin:{os.environ.get("PATH", "")}"
try:
    import torch
    cap = torch.cuda.get_device_capability()
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cap[0]}.{cap[1]}"
except Exception:
    os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0;8.9"
os.environ["BUILD_TARGET"] = "cuda"
os.environ["MAX_JOBS"] = "4"

# 1. Install Python 3.12 C headers and ninja
!apt-get update -qq && apt-get install -y -qq python3.12-dev ninja-build
!uv pip install --python /content/Effigy/.venv-ml/bin/python zstandard plyfile

# 2. Bypass PyTorch nvcc version check
cpp_ext = "/content/Effigy/.venv-ml/lib/python3.12/site-packages/torch/utils/cpp_extension.py"
if os.path.exists(cpp_ext):
    with open(cpp_ext, "r") as f:
        code = f.read()
    if "def _check_cuda_version(compiler_name: str, compiler_version: TorchVersion) -> None:" in code:
        code = code.replace(
            "def _check_cuda_version(compiler_name: str, compiler_version: TorchVersion) -> None:",
            "def _check_cuda_version(compiler_name: str, compiler_version: TorchVersion) -> None:\n    return",
        )
        with open(cpp_ext, "w") as f:
            f.write(code)
        print("✅ Patched torch cpp_extension._check_cuda_version")

# 3. Clone source repos into /tmp/build_wheels if not present
os.makedirs("/tmp/build_wheels", exist_ok=True)

if not os.path.exists("/tmp/build_wheels/cumesh"):
    !git clone --filter=blob:none https://github.com/JeffreyXiang/CuMesh /tmp/build_wheels/cumesh
    !git -C /tmp/build_wheels/cumesh checkout 12289e1062f0603f2f0d0771b02e1395d247f26f
    !git -C /tmp/build_wheels/cumesh submodule update --init --recursive

if not os.path.exists("/tmp/build_wheels/flex_gemm"):
    !git clone --filter=blob:none https://github.com/JeffreyXiang/FlexGEMM /tmp/build_wheels/flex_gemm
    !git -C /tmp/build_wheels/flex_gemm checkout 6dd94a859c26ee8246888502eada3dd8ad85532e
    !git -C /tmp/build_wheels/flex_gemm submodule update --init --recursive
    if os.path.exists("/content/Effigy/engine/trellis2/flex-gemm-windows.patch"):
        !git -C /tmp/build_wheels/flex_gemm apply /content/Effigy/engine/trellis2/flex-gemm-windows.patch

if not os.path.exists("/tmp/build_wheels/trellis2"):
    !git clone --filter=blob:none https://github.com/microsoft/TRELLIS.2 /tmp/build_wheels/trellis2
    !git -C /tmp/build_wheels/trellis2 checkout 75fbf0183001ed9876c8dbb35de6b68552ee08bd
    !git -C /tmp/build_wheels/trellis2 submodule update --init --recursive
    if os.path.exists("/content/Effigy/engine/trellis2/o-voxel-windows.patch"):
        !git -C /tmp/build_wheels/trellis2 apply /content/Effigy/engine/trellis2/o-voxel-windows.patch

# 4. Compile and install cumesh, flex_gemm, and o_voxel into .venv-ml
for name, pkg_dir in [
    ("cumesh", "/tmp/build_wheels/cumesh"),
    ("flex_gemm", "/tmp/build_wheels/flex_gemm"),
    ("o_voxel", "/tmp/build_wheels/trellis2/o-voxel"),
]:
    print(f"\n🔨 Building & installing {name} into .venv-ml...")
    res = subprocess.run(
        ["/content/Effigy/.venv-ml/bin/python", "setup.py", "install"],
        cwd=pkg_dir,
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print(f"❌ {name} build failed:\n", res.stdout[-2000:], "\n", res.stderr[-2000:])
        break
    else:
        print(f"✅ {name} installed successfully!")

# 5. Verify
!/content/Effigy/.venv-ml/bin/python -c "import cumesh, flex_gemm; print('\n🎉 cumesh & flex_gemm are ready for 3D generation!')"


In [ ]:
# Step 3: Install cloudflared for public tunneling without signup
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Step 4: Start Effigy server and print public access URL
import subprocess, time, re, os, threading, urllib.request

# 1. Clean up any existing instances and clear port 8000
!fuser -k 8000/tcp 2>/dev/null || true
!pkill -9 -f "server|uvicorn|cloudflared" 2>/dev/null || true
time.sleep(2)

# 2. Ensure server/agent.py permits remote tunnel requests (fixes 403 bad_origin blank screen)
agent_file = "/content/Effigy/server/agent.py"
if os.path.exists(agent_file):
    with open(agent_file, "r") as f:
        src = f.read()
    if "EFFIGY_ALLOW_REMOTE" not in src:
        src = "import os\n" + src.replace(
            "async def __call__(self, scope, receive, send):",
            "async def __call__(self, scope, receive, send):\n        if os.environ.get(\"EFFIGY_ALLOW_REMOTE\") == \"1\":\n            await self.app(scope, receive, send)\n            return"
        )
        with open(agent_file, "w") as f:
            f.write(src)
        print("✅ Configured server for remote tunnel access")

# Ensure MIN_SM allows A100 (SM 8.0) and socket SO_REUSEADDR
machine_file = "/content/Effigy/server/machine.py"
if os.path.exists(machine_file):
    with open(machine_file, "r") as f:
        m_src = f.read()
    m_src = m_src.replace("MIN_SM = 8.9", "MIN_SM = 8.0")
    if "SO_REUSEADDR" not in m_src:
        m_src = m_src.replace(
            "with socket.socket() as s:\n        try:\n            s.bind((\"127.0.0.1\", port))",
            "with socket.socket() as s:\n        try:\n            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n            s.bind((\"127.0.0.1\", port))"
        )
    with open(machine_file, "w") as f:
        f.write(m_src)
    print("✅ Enabled A100 (SM 8.0) and SO_REUSEADDR in machine.py")

# Bypass 16GB laptop RAM pre-check on 167GB server
presets_file = "/content/Effigy/server/presets.py"
if os.path.exists(presets_file):
    with open(presets_file, "r") as f:
        p_src = f.read()
    if "EFFIGY_IGNORE_RAM_CHECK" not in p_src:
        p_src = p_src.replace(
            "def ram_gb() -> tuple[float, float]:",
            "def ram_gb() -> tuple[float, float]:\n    if os.environ.get(\"EFFIGY_IGNORE_RAM_CHECK\") == \"1\":\n        return 167.0, 100.0"
        )
        with open(presets_file, "w") as f:
            f.write(p_src)
    print("✅ Disabled laptop RAM pre-check")

# 3. Start backend
print("Starting Effigy backend on A100...")
env = os.environ.copy()
env["EFFIGY_NO_BROWSER"] = "1"
env["EFFIGY_ALLOW_REMOTE"] = "1"
env["EFFIGY_IGNORE_RAM_CHECK"] = "1"
server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    ["uv", "run", "python", "-m", "server"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    cwd="/content/Effigy",
    env=env
)

print("Waiting for backend health check at http://127.0.0.1:8000/api/health...")
ready = False
for i in range(30):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                print(f"✅ Backend is live and healthy (ready in {i+1}s)!")
                break
    except Exception:
        pass

if not ready:
    print("❌ Backend failed to start. Last log lines from /content/server.log:")
    !tail -n 25 /content/server.log
else:
    # 4. Launch Cloudflare Tunnel and keep pipe alive
    print("Opening Cloudflare Tunnel...")
    tunnel_log = open("/content/tunnel.log", "w")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--http-host-header", "localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    url = None
    start = time.time()
    while time.time() - start < 30:
        line = tunnel_proc.stdout.readline()
        if not line:
            break
        tunnel_log.write(line)
        tunnel_log.flush()
        if "trycloudflare.com" in line:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                url = match.group(0)
                break
    def drain(pipe, log):
        for l in pipe:
            log.write(l)
    threading.Thread(target=drain, args=(tunnel_proc.stdout, tunnel_log), daemon=True).start()
    if url:
        print("\n" + "="*60)
        print("🚀 YOUR LIVE EFFIGY URL IS READY (VERIFIED HEALTHY):")
        print(f"{url}/#/library")
        print("="*60 + "\n")
    else:
        print("❌ Tunnel failed to produce a URL. Server log:")
        !tail -n 20 /content/server.log


In [ ]:
# (Optional) Step 5: Launch Fooocus (lllyasviel/Fooocus) in an Isolated Python 3.10 uv Environment
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
if not os.path.exists("/content/Fooocus"):
    !git clone https://github.com/lllyasviel/Fooocus.git /content/Fooocus
%cd /content/Fooocus
!git reset --hard HEAD
!rm -f /content/Fooocus/empty_reqs.txt

# Create isolated Python 3.10 venv via uv (Colab's default Python 3.13 breaks Gradio 3.41.2)
!uv venv /content/fooocus-venv --python 3.10 --seed
!uv pip install --python /content/fooocus-venv/bin/python torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/fooocus-venv/bin/python -r requirements_versions.txt fastapi==0.103.2 starlette==0.27.0 "pydantic>=1.10,<2.0" uvicorn==0.23.2
!/content/fooocus-venv/bin/python entry_with_update.py --share --always-high-vram
